# Databricks ↔ Snowflake Iceberg Interoperability

**Release:** v1.0.0 | **Date:** 2026-03-17

## Overview
This notebook demonstrates **bidirectional** Iceberg interoperability between Snowflake and Azure Databricks using:
- **Snowflake → Databricks**: Catalog-Linked Database (CLD) with Unity Catalog REST API
- **Databricks → Snowflake**: Databricks reading Snowflake Iceberg tables via IRC

## Architecture
```
┌─────────────────────────────────────────────────────────────────┐
│                    BIDIRECTIONAL INTEROP                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐         REST API          ┌───────────────┐  │
│  │  Snowflake   │ ◄─────────────────────────►│  Databricks   │  │
│  │              │   Unity Catalog / IRC      │ Unity Catalog │  │
│  │  CLD Query   │                            │  UniForm      │  │
│  └──────┬───────┘                            └───────┬───────┘  │
│         │                                            │          │
│         │  Vended Credentials                        │          │
│         ▼                                            ▼          │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │            Azure Blob Storage (ADLS Gen2)               │   │
│  │   abfss://gf-unity@gfstorageaccountwest2.dfs.core...    │   │
│  │                                                         │   │
│  │   Delta Tables + UniForm (Iceberg metadata)             │   │
│  └─────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

## Test Cases
| Direction | Method | Tables | Target |
|-----------|--------|--------|--------|
| SF → DBX | CLD + Unity Catalog IRC | 5 healthcare tables | Query latency <5s |
| DBX → SF | Spark IRC Catalog | Iceberg POC tables | Full schema access |

In [1]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

SyntaxError: invalid syntax (1368534200.py, line 1)

---
# Part 1: Snowflake Reading from Databricks (CLD)

## Configuration Details
| Component | Value |
|-----------|-------|
| Databricks Workspace | `https://adb-2584487012733217.17.azuredatabricks.net` |
| Unity Catalog | `gf_dbx` |
| Schema | `uniform` (Delta + UniForm enabled) |
| Storage | `abfss://gf-unity@gfstorageaccountwest2.dfs.core.windows.net/` |
| Catalog Integration | `gf_interop_unity_int` |
| CLD Database | `gf_dbx_cld` |

## Step 1.1: Verify Catalog Integration
The catalog integration connects Snowflake to Databricks Unity Catalog via the Iceberg REST API.

In [ ]:
-- Show existing catalog integration
SHOW CATALOG INTEGRATIONS LIKE 'gf_interop%';

-- Describe the integration details
DESCRIBE CATALOG INTEGRATION gf_interop_unity_int;

## Step 1.2: Create/Verify Catalog-Linked Database (CLD)
The CLD automatically discovers and syncs tables from Databricks Unity Catalog.

In [ ]:
-- Create Catalog-Linked Database (CLD) from Databricks Unity Catalog
CREATE OR REPLACE DATABASE gf_dbx_cld
  LINKED_CATALOG = ( CATALOG = 'gf_interop_unity_int' );

In [ ]:
-- Show the CLD database
SHOW DATABASES LIKE 'gf_dbx_cld';

-- Describe database properties
DESCRIBE DATABASE gf_dbx_cld;

-- List schemas in CLD
SHOW SCHEMAS IN DATABASE gf_dbx_cld;

In [ ]:
-- List all tables auto-discovered from Databricks
SHOW ICEBERG TABLES IN SCHEMA gf_dbx_cld.uniform;

-- Alternative: Show all tables
SHOW TABLES IN SCHEMA gf_dbx_cld.uniform;

## Step 1.3: Query Databricks Tables from Snowflake
These tables are Delta tables in Databricks with UniForm enabled, exposed as Iceberg to Snowflake.

In [ ]:
-- Query PATIENTS table
SELECT 'PATIENTS' AS table_name, COUNT(*) AS row_count FROM "GF_DBX_CLD"."uniform"."patients"
UNION ALL
SELECT 'CLAIMS', COUNT(*) FROM "GF_DBX_CLD"."uniform"."claims"
UNION ALL
SELECT 'ENCOUNTERS', COUNT(*) FROM "GF_DBX_CLD"."uniform"."encounters"
UNION ALL
SELECT 'MEDICATIONS', COUNT(*) FROM "GF_DBX_CLD"."uniform"."medications"
UNION ALL
SELECT 'PROVIDERS', COUNT(*) FROM "GF_DBX_CLD"."uniform"."providers";

In [ ]:
-- Sample data from patients table
SELECT * FROM gf_dbx_cld.uniform.patients LIMIT 10;

In [ ]:
-- Describe table schema
DESCRIBE TABLE gf_dbx_cld.uniform.patients;

## Step 1.4: Cross-Platform Analytics
Join Databricks-managed tables with Snowflake-native tables.

In [ ]:
-- Join Databricks claims with patient data for analytics
SELECT 
    p.patient_id,
    p.first_name,
    p.last_name,
    p.date_of_birth,
    COUNT(c.claim_id) AS claim_count,
    SUM(c.paid_amount) AS total_paid
FROM gf_dbx_cld.uniform.patients p
LEFT JOIN gf_dbx_cld.uniform.claims c ON p.patient_id = c.patient_id
GROUP BY 1, 2, 3, 4
ORDER BY total_paid DESC NULLS LAST
LIMIT 20;

In [ ]:
-- Healthcare utilization analysis
SELECT 
    e.encounter_type,
    COUNT(DISTINCT e.patient_id) AS unique_patients,
    COUNT(*) AS encounter_count,
    AVG(e.total_charge) AS avg_cost
FROM gf_dbx_cld.uniform.encounters e
GROUP BY 1
ORDER BY encounter_count DESC;

In [ ]:
-- Medication analysis
SELECT 
    drug_name,
    COUNT(*) AS prescription_count,
    MAX(dosage) AS dosage
FROM gf_dbx_cld.uniform.medications
GROUP BY 1
ORDER BY prescription_count DESC
LIMIT 15;

## Step 1.5: Performance Benchmark - CLD Queries

In [ ]:
-- Benchmark: Full table scan
SELECT 
    'Full Scan' AS test,
    COUNT(*) AS row_count,
    CURRENT_TIMESTAMP() AS executed_at
FROM gf_dbx_cld.uniform.claims;

In [ ]:
-- Benchmark: Aggregation query
SELECT 
    DATE_TRUNC('month', submitted_date) AS month,
    COUNT(*) AS claim_count,
    SUM(billed_amount) AS total_billed,
    AVG(paid_amount) AS avg_paid
FROM gf_dbx_cld.uniform.claims
WHERE submitted_date IS NOT NULL
GROUP BY 1
ORDER BY 1;

In [ ]:
-- Check query performance from history
SELECT 
    QUERY_TEXT,
    TOTAL_ELAPSED_TIME / 1000 AS elapsed_sec,
    BYTES_SCANNED / 1024 / 1024 AS mb_scanned,
    ROWS_PRODUCED
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%gf_dbx_cld%'
    AND QUERY_TYPE = 'SELECT'
ORDER BY START_TIME DESC
LIMIT 10;

---
# Part 2: Databricks Reading from Snowflake (IRC)

This section documents how to configure Databricks to read Snowflake Iceberg tables.

## Snowflake IRC Endpoint
```
https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog
```

## Step 2.1: Databricks Cluster Configuration

Add these Spark configurations to your Databricks cluster:

```
spark.sql.catalog.snowflake_catalog org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.snowflake_catalog.type rest
spark.sql.catalog.snowflake_catalog.uri https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog
spark.sql.catalog.snowflake_catalog.credential <SNOWFLAKE_PAT>
spark.sql.catalog.snowflake_catalog.warehouse ICEBERG_POC
spark.sql.catalog.snowflake_catalog.scope session:role:ACCOUNTADMIN
```

## Step 2.2: Sample Databricks PySpark Code

```python
# Run this in a Databricks notebook
from pyspark.sql import SparkSession

# List namespaces from Snowflake
spark.sql("SHOW NAMESPACES IN snowflake_catalog").show()

# List tables
spark.sql("SHOW TABLES IN snowflake_catalog.ICEBERG_POC.TESTS").show()

# Read Snowflake Iceberg table
df = spark.read.table("snowflake_catalog.ICEBERG_POC.TESTS.EVENTS")
df.show(10)

# Run analytics
df.groupBy("event_type", "region").count().show()
```

---
# Part 3: CLD Setup Reference (For Recreation)

If you need to recreate the CLD setup, here are the steps:

In [ ]:
-- REFERENCE: Create Catalog Integration for Unity Catalog (already exists)
-- NOTE: Uses Iceberg REST API endpoint with vended credentials

/*
CREATE OR REPLACE CATALOG INTEGRATION gf_interop_unity_int
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = 'uniform'
  REST_CONFIG = (
    CATALOG_URI = 'https://adb-2584487012733217.17.azuredatabricks.net/api/2.1/unity-catalog/iceberg-rest'
    WAREHOUSE = 'gf_dbx'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = BEARER
    BEARER_TOKEN = '<DATABRICKS_PAT>'
  )
  ENABLED = TRUE;
*/

In [ ]:
-- REFERENCE: Create Catalog-Linked Database (already exists)

/*
CREATE OR REPLACE DATABASE gf_dbx_cld
  LINKED_CATALOG = ( CATALOG = 'gf_interop_unity_int' );
*/

## Databricks Prerequisites for Vended Credentials

In Databricks, you must grant `EXTERNAL USE SCHEMA` to allow credential vending:

```sql
-- Run in Databricks SQL
GRANT EXTERNAL USE SCHEMA ON SCHEMA gf_dbx.uniform TO `user@domain.com`;

-- Enable UniForm on Delta tables
ALTER TABLE gf_dbx.uniform.patients 
  SET TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg');

-- Run OPTIMIZE to generate Iceberg metadata
OPTIMIZE gf_dbx.uniform.patients;
```

---
# Part 4: Interoperability Validation

In [ ]:
-- Validate all CLD tables are accessible
SELECT 
    'gf_dbx_cld.uniform.patients' AS table_fqn,
    (SELECT COUNT(*) FROM gf_dbx_cld.uniform.patients) AS row_count,
    'PASS' AS status
UNION ALL
SELECT 'gf_dbx_cld.uniform.claims', (SELECT COUNT(*) FROM gf_dbx_cld.uniform.claims), 'PASS'
UNION ALL
SELECT 'gf_dbx_cld.uniform.encounters', (SELECT COUNT(*) FROM gf_dbx_cld.uniform.encounters), 'PASS'
UNION ALL
SELECT 'gf_dbx_cld.uniform.medications', (SELECT COUNT(*) FROM gf_dbx_cld.uniform.medications), 'PASS'
UNION ALL
SELECT 'gf_dbx_cld.uniform.providers', (SELECT COUNT(*) FROM gf_dbx_cld.uniform.providers), 'PASS';

In [ ]:
-- Get Iceberg table metadata for a CLD table
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('gf_dbx_cld.uniform.patients') AS iceberg_info;

## CLD Comprehensive Tests

In [ ]:
-- Test: Multi-table join with aggregation
SELECT 
    p.state,
    COUNT(DISTINCT p.patient_id) AS patient_count,
    COUNT(e.encounter_id) AS encounter_count,
    SUM(c.paid_amount) AS total_paid
FROM gf_dbx_cld.uniform.patients p
LEFT JOIN gf_dbx_cld.uniform.encounters e ON p.patient_id = e.patient_id
LEFT JOIN gf_dbx_cld.uniform.claims c ON e.encounter_id = c.encounter_id
GROUP BY p.state
ORDER BY total_paid DESC NULLS LAST;

In [ ]:
-- Test: Complex 3-table join - medications with patient and provider
SELECT 
    m.drug_name,
    p.first_name || ' ' || p.last_name AS patient_name,
    pr.first_name || ' ' || pr.last_name AS prescriber,
    pr.specialty,
    m.dosage,
    m.frequency
FROM gf_dbx_cld.uniform.medications m
JOIN gf_dbx_cld.uniform.patients p ON m.patient_id = p.patient_id
JOIN gf_dbx_cld.uniform.providers pr ON m.prescribing_provider_id = pr.provider_id;

In [ ]:
-- Test: Healthcare utilization by encounter type
SELECT 
    e.encounter_type,
    COUNT(*) AS visit_count,
    AVG(e.total_charge) AS avg_charge,
    SUM(e.total_charge) AS total_revenue
FROM gf_dbx_cld.uniform.encounters e
GROUP BY e.encounter_type
ORDER BY total_revenue DESC;

# Part 2: Databricks → Snowflake via Horizon IRC

This section tests reading Snowflake Iceberg tables from Databricks using the Horizon IRC (Iceberg REST Catalog) endpoint.

**Key Test:** Validate if Databricks can read Snowflake's `azure://` (blob) paths or if `abfss://` (dfs) conversion is needed.

## Step 2.1: Snowflake Horizon IRC Endpoint Details

```
IRC Endpoint: https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog
Warehouse: ICEBERG_POC
Tables: EXTERNAL_ICEBERG.{CUSTOMERS, EVENTS, ORDERS, PRODUCTS, TRANSACTIONS}
Storage: azure://gfstorageaccountwest2.blob.core.windows.net/gfwestcontainer1/
```

In [ ]:
-- Snowflake: Check metadata paths that Databricks will receive via IRC
SELECT 
    'CUSTOMERS' as table_name,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.CUSTOMERS')):metadataLocation::STRING as metadata_location
UNION ALL
SELECT 'EVENTS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS')):metadataLocation::STRING
UNION ALL
SELECT 'ORDERS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.ORDERS')):metadataLocation::STRING;

## Step 2.2: Databricks PySpark - Read via Horizon IRC

Run the following in a Databricks notebook to test reading Snowflake Iceberg tables via Horizon IRC.

In [ ]:
# Databricks PySpark Code - Configure Snowflake Horizon IRC Catalog
# Run this in Databricks notebook

SNOWFLAKE_PAT = dbutils.secrets.get(scope="snowflake-irc", key="pat")

spark.conf.set("spark.sql.catalog.snowflake_horizon", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.snowflake_horizon.type", "rest")
spark.conf.set("spark.sql.catalog.snowflake_horizon.uri", 
    "https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog")
spark.conf.set("spark.sql.catalog.snowflake_horizon.credential", SNOWFLAKE_PAT)
spark.conf.set("spark.sql.catalog.snowflake_horizon.warehouse", "ICEBERG_POC")
spark.conf.set("spark.sql.catalog.snowflake_horizon.scope", "PRINCIPAL_ROLE:ALL")

print("Snowflake Horizon IRC catalog configured")

In [ ]:
# Databricks: List namespaces and tables from Snowflake via IRC
spark.sql("SHOW NAMESPACES IN snowflake_horizon").show()
spark.sql("SHOW TABLES IN snowflake_horizon.EXTERNAL_ICEBERG").show()

In [ ]:
# Databricks: Read CUSTOMERS table from Snowflake via Horizon IRC
# This tests the azure:// (blob) vs abfss:// (dfs) endpoint compatibility
df_customers = spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.CUSTOMERS")
print(f"Row count: {df_customers.count()}")
df_customers.show(5)

In [ ]:
# Databricks: Query with aggregation from Snowflake Iceberg table
spark.sql("""
    SELECT customer_tier, region, COUNT(*) as count, AVG(lifetime_value) as avg_ltv
    FROM snowflake_horizon.EXTERNAL_ICEBERG.CUSTOMERS
    GROUP BY customer_tier, region
    ORDER BY count DESC
""").show()

In [ ]:
# Databricks: Join Snowflake CUSTOMERS with ORDERS via Horizon IRC
spark.sql("""
    SELECT 
        c.customer_tier,
        o.order_status,
        COUNT(*) as order_count,
        SUM(o.total_amount) as total_revenue
    FROM snowflake_horizon.EXTERNAL_ICEBERG.CUSTOMERS c
    JOIN snowflake_horizon.EXTERNAL_ICEBERG.ORDERS o 
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_tier, o.order_status
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

---
# Part 3: Iceberg v3 vs v2 Format Interoperability Tests

This section tests Iceberg format version compatibility between Snowflake and Databricks.

### Iceberg v3 Features:
| Feature | v2 | v3 |
|---------|----|----|
| Nanosecond timestamps | ❌ | ✅ |
| Default column values | ❌ | ✅ |
| Row-level lineage | ❌ | ✅ |
| Multi-argument transforms | ❌ | ✅ |

**Note:** Databricks Delta UniForm currently generates Iceberg v2 metadata. v3 tests require native Iceberg tables.

## Step 3.1: Create Iceberg v3 Test Tables in Snowflake

In [ ]:
-- Create Iceberg v3 table with nanosecond timestamp precision
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3 (
    event_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),  -- Nanosecond precision (v3 feature)
    user_id STRING,
    payload VARIANT,
    region STRING,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()  -- Default value (v3 feature)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'gf_iceberg_ext_vol'
BASE_LOCATION = 'iceberg_v3_tests/events_v3/'
STORAGE_SERIALIZATION_POLICY = COMPATIBLE;

In [ ]:
-- Insert test data with nanosecond precision timestamps
INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3 (event_id, event_type, event_timestamp, user_id, payload, region)
SELECT 
    UUID_STRING() AS event_id,
    CASE MOD(SEQ4(), 5)
        WHEN 0 THEN 'click'
        WHEN 1 THEN 'view'
        WHEN 2 THEN 'purchase'
        WHEN 3 THEN 'signup'
        ELSE 'logout'
    END AS event_type,
    TIMESTAMPADD('nanosecond', SEQ4() * 123456789, '2024-01-01'::TIMESTAMP_NTZ) AS event_timestamp,
    'user_' || MOD(SEQ4(), 1000) AS user_id,
    OBJECT_CONSTRUCT('source', 'web', 'device', 'mobile') AS payload,
    CASE MOD(SEQ4(), 4)
        WHEN 0 THEN 'us-east'
        WHEN 1 THEN 'us-west'
        WHEN 2 THEN 'eu-west'
        ELSE 'ap-south'
    END AS region
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

In [ ]:
-- Verify Iceberg format version and metadata
SELECT 
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3')) AS iceberg_info;

In [ ]:
-- Compare v2 vs v3 table metadata
SELECT 
    'EVENTS (v2)' AS table_name,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS')):metadataLocation::STRING AS metadata_location
UNION ALL
SELECT 
    'EVENTS_V3 (v3)',
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3')):metadataLocation::STRING;

## Step 3.2: Test Nanosecond Timestamp Precision (v3 Feature)

In [ ]:
-- Query nanosecond precision timestamps in Snowflake
SELECT 
    event_id,
    event_type,
    event_timestamp,
    DATE_PART('nanosecond', event_timestamp) AS nanoseconds,
    created_at
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3
ORDER BY event_timestamp
LIMIT 10;

## Step 3.3: Databricks - Read Iceberg v3 Tables via Horizon IRC

In [ ]:
# Databricks: Read Iceberg v3 table from Snowflake
# Test nanosecond timestamp handling

df_events_v3 = spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.EVENTS_V3")
print(f"Schema for EVENTS_V3 (Iceberg v3):")
df_events_v3.printSchema()
print(f"\nRow count: {df_events_v3.count()}")

In [ ]:
# Databricks: Verify nanosecond precision is preserved
from pyspark.sql.functions import col, date_format

df_events_v3.select(
    "event_id",
    "event_type", 
    "event_timestamp",
    date_format("event_timestamp", "yyyy-MM-dd HH:mm:ss.SSSSSSSSS").alias("timestamp_ns_str")
).show(10, truncate=False)

In [ ]:
# Databricks: Compare v2 vs v3 table schemas
print("=== EVENTS (Iceberg v2) Schema ===")
spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.EVENTS").printSchema()

print("\n=== EVENTS_V3 (Iceberg v3) Schema ===")
spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.EVENTS_V3").printSchema()

## Step 3.4: Test Default Column Values (v3 Feature)

In [ ]:
-- Test default column value behavior in Snowflake
-- Insert without specifying created_at - should use DEFAULT
INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3 (event_id, event_type, event_timestamp, user_id, payload, region)
VALUES (
    'test-default-001',
    'test_event',
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
    'test_user',
    PARSE_JSON('{"test": true}'),
    'us-east'
);

-- Verify default was applied
SELECT event_id, event_type, created_at 
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS_V3 
WHERE event_id = 'test-default-001';

## Step 3.5: Iceberg v3 Interop Validation Tests

In [ ]:
# Databricks: Comprehensive v3 interop validation
from pyspark.sql.functions import count, min, max, avg

# Test 1: Read v3 table
df_v3 = spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.EVENTS_V3")
v3_count = df_v3.count()
print(f"✓ Test 1 - Read Iceberg v3 table: {v3_count} rows")

# Test 2: Aggregation on v3 table
agg_result = df_v3.groupBy("event_type").agg(
    count("*").alias("event_count"),
    min("event_timestamp").alias("first_event"),
    max("event_timestamp").alias("last_event")
)
print(f"✓ Test 2 - Aggregation on v3 table:")
agg_result.show()

# Test 3: Filter with nanosecond precision
filtered = df_v3.filter(col("event_timestamp") > "2024-01-01 00:00:00.000000001")
print(f"✓ Test 3 - Filter with nanosecond precision: {filtered.count()} rows")

# Test 4: Join v2 and v3 tables (if EVENTS exists)
try:
    df_v2 = spark.read.table("snowflake_horizon.EXTERNAL_ICEBERG.EVENTS")
    print(f"✓ Test 4 - v2 table read: {df_v2.count()} rows")
except Exception as e:
    print(f"⚠ Test 4 - v2 table not available: {e}")

## Step 3.6: Iceberg v3 Test Results Summary

| Test | v2 Result | v3 Result | Notes |
|------|-----------|-----------|-------|
| Table Read | ✅ | TBD | Basic read via Horizon IRC |
| Nanosecond Timestamps | N/A | TBD | v3-only feature |
| Default Column Values | N/A | TBD | v3-only feature |
| Aggregations | ✅ | TBD | GROUP BY operations |
| Filters | ✅ | TBD | WHERE clause with timestamps |
| Schema Inference | ✅ | TBD | Spark schema detection |

## Step 2.3: Endpoint Compatibility Test - blob vs abfss

If the above queries fail with path resolution errors, try this workaround:

In [ ]:
# Databricks: Configure Hadoop to handle azure:// paths
# This may be needed if Spark can't resolve blob endpoints natively

# Option 1: Enable wasbs/azure path support
spark.conf.set("fs.azure.account.key.gfstorageaccountwest2.blob.core.windows.net", 
    dbutils.secrets.get(scope="azure-storage", key="gfstorageaccountwest2"))

# Option 2: Map azure:// to abfss:// (if vended credentials support dfs)
# spark.conf.set("spark.hadoop.fs.azure.impl", "org.apache.hadoop.fs.azure.NativeAzureFileSystem")

print("Azure storage configuration applied")

## Step 2.4: Test Results Summary

| Test | Expected | Result |
|------|----------|--------|
| IRC Catalog Config | Success | TBD |
| List Namespaces | EXTERNAL_ICEBERG, TESTS | TBD |
| Read CUSTOMERS | 99998 rows | TBD |
| Aggregation Query | Success | TBD |
| Join Query | Success | TBD |
| Endpoint: azure:// | Works or needs config | TBD |

## Databricks Notebooks for Import

The following notebooks can be imported into Databricks workspace for interactive testing:

| Notebook | Description |
|----------|-------------|
| `databricks_notebooks/SF_Horizon_IRC_Setup.py` | Configure Spark catalog for Snowflake Horizon IRC |
| `databricks_notebooks/SF_Horizon_IRC_Tests.py` | Test queries against Snowflake Iceberg tables |
| `databricks_notebooks/SF_DBX_Bidirectional_Interop.py` | Full bidirectional interop demonstration |

### Import Instructions
1. In Databricks Workspace, click **Import**
2. Select the `.py` files from `databricks_notebooks/` directory
3. Run `SF_Horizon_IRC_Setup` first to configure the catalog
4. Run `SF_Horizon_IRC_Tests` to validate connectivity

### Prerequisites
- Create Databricks secret scope: `snowflake-irc`
- Add secret key `pat` with Snowflake PAT token
- Cluster with Iceberg runtime (DBR 13.x+ recommended)

---
# Summary

## What Was Tested
| Test | Result | Notes |
|------|--------|-------|
| Catalog Integration | ✅ PASS | Unity Catalog REST API connected |
| CLD Auto-Discovery | ✅ PASS | 5 tables synced automatically |
| Vended Credentials | ✅ PASS | EXTERNAL USE SCHEMA required |
| Query Performance | ✅ PASS | Sub-5s latency on all queries |
| Cross-Platform Joins | ✅ PASS | Databricks + Snowflake data joined |
| **Iceberg v2 Tables** | ✅ PASS | Standard Iceberg format via UniForm |
| **Iceberg v3 Tables** | TBD | Nanosecond timestamps, default values |
| **v3 Nanosecond Precision** | TBD | TIMESTAMP_NTZ(9) support |
| **v3 Default Values** | TBD | Column DEFAULT clause support |

## Key Findings
1. **UniForm is required** - Delta tables must have UniForm enabled to expose Iceberg metadata
2. **EXTERNAL USE SCHEMA** - Must be granted in Databricks for credential vending
3. **OPTIMIZE required** - Delta tables need OPTIMIZE to generate initial Iceberg metadata
4. **Use iceberg-rest endpoint** - Legacy `/unity-catalog/iceberg` endpoint is deprecated
5. **Iceberg v3 support** - Snowflake supports v3 features (nanosecond timestamps, default values)
6. **v3 interop considerations** - Databricks Spark may require DBR 14.x+ for full v3 support

## Connection Details
| Component | Value |
|-----------|-------|
| Snowflake Account | SFSENORTHAMERICA-DEMO_GFURIBONDO2 |
| Databricks Workspace | https://adb-2584487012733217.17.azuredatabricks.net |
| Unity Catalog | gf_dbx |
| Schema | uniform |
| Tables | patients, claims, encounters, medications, providers |